In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import warnings

import polars as pl
from maite_datasets.object_detection import SeaDrone

from dataeval import Metadata
from dataeval.data import Indices, Limit, Shuffle, View
from dataeval.exceptions import MetadataFormatError

In [3]:
# This guide does not use the binned factors, so skip the warnings about binning them.
warnings.filterwarnings("ignore", message=".*(was dropped|binned automatically).*")

In [4]:
dataset = View(SeaDrone(root="./data", image_set="val", download=True), [Shuffle(seed=0), Limit(200)])
metadata = Metadata(dataset)
high = metadata.where(pl.col("altitude") > 60, level="unit")

print("before:", metadata.level_counts)
print("after :", high.level_counts)
print("is_filtered:", high.is_filtered)

before: {'unit': 200, 'instance': 1305}
after : {'unit': 22, 'instance': 192}
is_filtered: True


In [5]:
items = high.selected_items()
matching_dataset = View(dataset, Indices(items.tolist()))

print("surviving items :", len(items))
print("matching dataset:", len(matching_dataset))

surviving items : 22
matching dataset: 22


In [6]:
swimmer_rows = metadata.where(pl.col("class_label") == 1, level="instance")
try:
    swimmer_rows.selected_items()
except ValueError as error:
    print("as expected:", str(error).split(".")[0])

as expected: This metadata was filtered below the item level, so no subset of the dataset reproduces it: some 'unit' row kept only part of its rows


In [7]:
metadata.save("./data/seadrone-metadata.dem")
reloaded = Metadata.load("./data/seadrone-metadata.dem", dataset)

print("levels :", reloaded.levels)
print("counts :", reloaded.level_counts)

levels : ('unit', 'instance')
counts : {'unit': 200, 'instance': 1305}


In [8]:
try:
    metadata = Metadata.load("./data/seadrone-metadata.dem", dataset)
except MetadataFormatError:
    metadata = Metadata(dataset)
    metadata.save("./data/seadrone-metadata.dem")

print("ready:", metadata.level_counts)

ready: {'unit': 200, 'instance': 1305}


In [9]:
# TEST ASSERTION CELL ###
assert high.is_filtered
assert high.level_counts["unit"] < metadata.level_counts["unit"]
assert len(items) == high.level_counts["unit"]
assert len(matching_dataset) == len(items)

assert reloaded.levels == metadata.levels
assert reloaded.level_counts == metadata.level_counts
assert reloaded.factor_names == metadata.factor_names
assert reloaded.rows_at("instance")["item_index"].to_list() == metadata.rows_at("instance")["item_index"].to_list()